# 06 — Physics model: what drives gas and water? + ablation of every modelling choice

Physics model (approach B, `src/vfm.py`):
```
q_oil   = J · (P_res(t) − Pwf)          J from Darcy (k, s from PTA),  P_res(t) fitted per well
q_water = q_oil · WC(t) / (100 − WC(t))  WC(t) fitted per well
q_gas   = q_oil · GOR / 1000             GOR per well
```
## 1. Are the gas / water errors predictable or just measurement noise?
We take each test's deviation from the well's GOR mean (and from the WC time trend) and check
(a) whether any feature correlates with it, (b) lag-1 autocorrelation in time: real slow drifts give
**positive** autocorrelation; random measurement noise gives ~0 or negative.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')

tt['gor_dev'] = tt.gor_scf_stb - tt.groupby('well_name').gor_scf_stb.transform('mean')
tt['wc_dev'] = np.nan
for well, g in tt.groupby('well_name'):
    trend = np.polyval(np.polyfit(g.days, g.water_cut_pct, 1), g.days)
    tt.loc[g.index, 'wc_dev'] = g.water_cut_pct - trend

cols = ['days', 'dh_pressure_psia', 'whp_psia', 'choke_opening_pct', 'choke_flow_term',
        'gas_lift_rate_mscfd', 'tubing_gradient', 'lift_dp', 'dh_temperature_degf']
within = tt[cols] - tt.groupby('well_name')[cols].transform('mean')
print('Within-well correlation with the deviations:')
print(pd.DataFrame({'GOR_dev': within.corrwith(tt.gor_dev), 'WC_dev': within.corrwith(tt.wc_dev)}).round(2))

for c in ['gor_dev', 'wc_dev']:
    ac = [g.sort_values('days')[c].autocorr() for _, g in tt.groupby('well_name')]
    print(f'{c} lag-1 autocorrelation per well:', np.round(ac, 2))

**Result:** GOR deviations correlate with nothing and flip sign test to test → **measurement noise**.
Physically consistent: Pwf (> 2,200 psia) stays above the bubble point (~1,300 psia), so no free gas forms
in the reservoir and the true GOR ≈ solution GOR = constant.
WC deviations: mostly noise (±1 point), small real link to tubing gradient (heavier column ↔ more water).

## 2. Ablation: change ONE choice at a time from the default

In [ ]:
grid = {
    'B default (darcy J, P_res lin, WC lin, GOR mean)': {},
    'B P_res constant': {'pres_deg': 0},
    'B P_res quadratic': {'pres_deg': 2},
    'B J fitted from tests': {'j_source': 'fitted'},
    'B WC constant': {'wc_model': 'wc_const'},
    'B WC quadratic': {'wc_model': 'wc_quadratic'},
    'B WOR linear': {'wc_model': 'wor_linear'},
    'B GOR median': {'gor_model': 'median'},
    'B GOR linear': {'gor_model': 'linear'},
    'B weight by test hours': {'weight_by_hours': True},
}
results = []
for name, options in grid.items():
    V.run_experiment(name, tt, vfm.make_physics_method(**options), results)

all_results = V.save_results(results, load.ROOT / 'outputs' / 'ablation_results.csv')
pd.DataFrame(results).round(2)

**Conclusions** (differences < ~0.3 % are within noise for 85 tests):
- Reservoir depletion (P_res falling in time) is essential: without it oil error 2.4 → 8.5 %.
- Water-cut trend is essential: without it water error 4.0 → 10.0 %. Quadratic overfits.
- Darcy J (from PTA k and skin) beats J fitted from 7–9 tests for oil.
- Constant mean GOR is best — matches the physics.
- → keep the default.

In [ ]:
all_results.sort_values('mean_MAPE').round(2)